# Hosted Tool Search

The Hidden Leaf Village has hundreds of mission tools. If we send every tool's details with every request, we pay for
all those tokens, even when the model only needs one tool. **Tool search** fixes this: most tools stay hidden, and the
model asks OpenAI's server to find the one it needs.

Three pieces make it work: `@function_tool(defer_loading=True)` hides a tool, `tool_namespace(...)` puts hidden tools in
a labeled group, and `ToolSearchTool()` is the tool the model uses to look inside the groups.

The search runs on OpenAI's computers, so a fake `ScriptedModel` cannot prove it. We check the setup offline, then
measure the real token savings live.


## Hide a tool until the model needs it

`defer_loading=True` is the whole trick: the tool exists, but its details are not sent until tool search finds it.


In [2]:
from agents import (
    FunctionTool,  # the tool object that @function_tool makes
    function_tool,  # turns a normal function into a tool an Agent can call
)


# Factory: builds one check_mission_NNN tool, hidden or not; called for all 200 missions below
def build_mission_tool(mission_num: int, deferred: bool) -> FunctionTool:
    # Define the tool
    # @function_tool callback: one mission's status lookup, hidden until tool search finds it when deferred
    @function_tool(
        name_override=f"check_mission_{mission_num:03d}",
        description_override=f"Check the status of Mission-{mission_num:03d}.",
        # defer_loading: True hides the tool until tool search finds it
        defer_loading=deferred,
    )
    def check_mission(danger_only: bool = False) -> str:
        return f"Mission-{mission_num:03d}: danger level 3, standard clear procedure."

    return check_mission

## Group the hidden tools and add `ToolSearchTool`

A namespace is a label plus a short description. The model reads only those short descriptions first, then loads the
one group it needs. So we split 200 tools into 10 small groups.


In [3]:
from agents import (
    ToolSearchTool,  # a tool that finds other hidden tools by their descriptions
    tool_namespace,  # groups related tools under one name and description
)


# Helper: builds 10 groups of 20 hidden tools, plus the one tool that searches them
def build_search_tools() -> list:
    tools: list = []
    # start goes 1, 21, 41, ... so each group holds one block of 20 missions
    for start in range(1, 201, 20):
        tools.extend(
            tool_namespace(
                name=f"missions_{start:03d}_{start + 19:03d}",
                description=f"Status tools for Hidden Leaf Village Missions {start:03d}-{start + 19:03d}.",
                tools=[build_mission_tool(number, deferred=True) for number in range(start, start + 20)],
            )
        )
    tools.append(ToolSearchTool())
    return tools

In [5]:
from agents import (
    Agent,  # bundles instructions, tools, and model settings into one agent
)

# Initialize the agent
search_agent = Agent(
    name="Leaf Village Assistant",
    instructions="Use tool search to find the right mission-status tool before answering. Answer briefly.",
    tools=build_search_tools(),
)
# proves every mission tool is hidden and the search tool is on the agent
hidden = [tool for tool in search_agent.tools if getattr(tool, "defer_loading", False)]
print("[hidden tools]", len(hidden))
print("[tool_search present]", any(tool.name == "tool_search" for tool in search_agent.tools))

[hidden tools] 200
[tool_search present] True


## Measure the token savings live — needs a real key

The same question runs twice: once with all 200 tools sent normally, once with the same 200 hidden in 10 groups. We
print the input tokens of each request. Skipped when there is no `OPENAI_API_KEY`.

**Gotcha, confirmed by measuring:** put all 200 tools in _one_ group and tool search can load every one of them at
once, so the saving is not steady. Several small groups are the dependable way. Token counts change between runs, so
treat them as a snapshot.


In [6]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    Agent,  # bundles instructions, tools, and model settings into one agent
    RunConfig,  # per-run overrides: model, workflow_name, group_id
    Runner,  # executes an Agent: run / run_sync / run_streamed
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# Demo: ask the same real question with 200 tools sent plainly, then hidden in ten groups
async def demo_token_savings() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    # Initialize the agent: all 200 tools sent with every request
    plain_agent = Agent(
        name="Leaf Village Assistant",
        instructions="Answer briefly.",
        tools=[build_mission_tool(number, deferred=False) for number in range(1, 201)],
    )
    for label, agent in [("plain", plain_agent), ("hidden", search_agent)]:
        # Run the Agent
        result = await Runner.run(
            agent,
            "What's the status of Mission-005?",
            run_config=RunConfig(workflow_name=f"e14-live-{label}"),
        )
        # proves tool search sends far fewer input tokens than sending all 200 tools every time
        input_tokens = [response.usage.input_tokens for response in result.raw_responses]
        print(f"[{label}] input tokens per request:", input_tokens)
        for item in result.new_items:
            if item.type == "tool_search_call_item":
                print(f"[{label} search call]", item.raw_item.arguments)
        print(f"[{label} final_output]", result.final_output)


await demo_token_savings()
# Outside a notebook: asyncio.run(demo_token_savings())

[plain] input tokens per request: [8637, 8686]
[plain final_output] Mission-005: **Danger level 3; standard clear procedure.**
[hidden] input tokens per request: [943, 1000]
[hidden search call] {'paths': ['missions_001_020'], 'query': '005'}
[hidden final_output] Mission-005: Danger level 3; standard clear procedure.


## Summary

- `@function_tool(defer_loading=True)` hides a tool, `tool_namespace(...)` groups hidden tools, and `ToolSearchTool()`
  lets the model find them, so you stop paying for tools it never uses.
- Split a big tool list into many small groups. One giant group can load everything at once and save nothing.
